# Civitas — silver translations for Model B (IndicTrans2)

Translates English Rewrite/Respond pairs into Hindi, Tamil, Telugu, Malayalam and
Roman Urdu (via Hindi, then romanised) so Model B learns every language.
Machine translation is imperfect: you review a sample afterwards.

## WHAT YOU DO
1. On your PC (repo root):
   `python -m model_b_generation.build_dataset_gen`, then
   `python -m model_b_generation.prepare_splits_gen`, then
   `python scripts/package_data.py --kind gen` → `artifacts/civitas_gen_data.zip`.
2. Upload `civitas_gen_data.zip` (and `civitas_code.zip` if you don't use GitHub) to **My Drive → civitas/**.
3. On huggingface.co, open `ai4bharat/indictrans2-en-indic-dist-200M` and accept its terms if
   it asks. Create a read token (Settings → Access Tokens) and add it in Colab under
   **🔑 Secrets** as `HF_TOKEN` (turn on notebook access).
4. **Runtime → T4 GPU → Run all.** About 30–60 min. Re-run after a disconnect: finished
   languages are skipped.
5. Download from Drive → `civitas/silver/`:
   - `silver_pairs.parquet` → `data/gen/silver_pairs.parquet`
   - `civitas_results_silver.zip` → `artifacts/`
6. Review ~100 pairs per language on your PC:
   `python -m model_b_generation.review_silver --language ta` (repeat for hi, te, ml, ur_roman).
7. Rebuild the splits with the silver data and re-package:
   `python -m model_b_generation.prepare_splits_gen --silver data/gen/silver_pairs.parquet`
   then `python scripts/package_data.py --kind gen`, upload the new zip, and run notebook 03.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
# Google Drive: everything that must survive a disconnect lives here
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/civitas"
import os
os.makedirs(PROJECT, exist_ok=True)
print("project folder:", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
# Get the code: from GitHub (fill REPO_URL) OR from civitas_code.zip in the Drive folder
REPO_URL = ""  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
import os, subprocess, zipfile, sys
CODE = "/content/civitas"
if REPO_URL:
    if os.path.exists(f"{CODE}/.git"):
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE], check=True)
else:
    # made on your PC with `make package-code`
    zpath = f"{PROJECT}/civitas_code.zip"
    assert os.path.exists(zpath), f"Set REPO_URL above, or upload civitas_code.zip to {PROJECT}"
    zipfile.ZipFile(zpath).extractall(CODE)
    inner = [d for d in os.listdir(CODE) if os.path.exists(f"{CODE}/{d}/config.py")]
    if not os.path.exists(f"{CODE}/config.py") and inner:  # zip had a top-level folder
        CODE = f"{CODE}/{inner[0]}"
os.chdir(CODE)
sys.path.insert(0, CODE)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
      or "code from zip")

In [ ]:
# IndicTrans2's remote code targets transformers 4.x, so pin it here (this runtime only)
!pip -q install "transformers>=4.40,<4.50" sentencepiece pyarrow "indic-transliteration>=2.3.45"
!pip -q install IndicTransToolkit || pip -q install git+https://github.com/VarunGumma/IndicTransToolkit
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("No HF_TOKEN secret (fine if the model is not gated):", e)

In [ ]:
# Unpack the data package and verify it against its manifest
import hashlib, json, zipfile, os
DATA = "/content/gen_data"
zipfile.ZipFile(f"{PROJECT}/civitas_gen_data.zip").extractall(DATA)
manifest = json.load(open(f"{DATA}/manifest.json"))
for name, digest in manifest["files"].items():
    h = hashlib.sha256(open(f"{DATA}/{name}", "rb").read()).hexdigest()
    assert h == digest, f"{name} is corrupted (sha256 mismatch) - re-upload the zip"
for split, info in manifest["splits"].items():
    print(f"{split:>5}: {info['rows']:>7} rows  languages={info['language_counts']}")
print("built from commit", manifest["git_commit"], "at", manifest["created_at"])

In [ ]:
# Config
PER_TASK = 2500   #@param {type:"integer"}  English rows per task (rewrite / respond) to translate
LANGS = ["hi", "ta", "te", "ml", "ur_roman"]
OUT = f"{PROJECT}/silver"
PARTS = f"{OUT}/parts"
os.makedirs(PARTS, exist_ok=True)
import time, pandas as pd
from model_b_generation.silver import select_english, make_silver_rows, IndicTrans2, INDICTRANS_TARGETS
pairs = pd.concat([pd.read_parquet(f"{DATA}/{s}.parquet") for s in ("train", "val", "test")])
english = select_english(pairs, PER_TASK)
print(english.groupby(["task", "split"]).size())
STARTED = time.time()

In [ ]:
# Translate, one language at a time. Each finished language is saved to Drive and skipped on re-run.
mt = None
cache = {}   # Hindi translations are reused for Roman Urdu
for lang in LANGS:
    part = f"{PARTS}/{lang}.parquet"
    if os.path.exists(part):
        print(lang, "already done"); continue
    tgt = INDICTRANS_TARGETS[lang]
    if tgt not in cache:
        mt = mt or IndicTrans2()
        t0 = time.time()
        texts = pd.unique(pd.concat([english["source_text"], english["target_text"]])).tolist()
        cache[tgt] = dict(zip(texts, mt.translate(texts, tgt)))
        print(f"{lang}: {len(texts)} sentences in {time.time() - t0:.0f}s")
    tr = cache[tgt]
    rows = make_silver_rows(english, lang, [tr[t] for t in english["source_text"]],
                            [tr[t] for t in english["target_text"]])
    rows.to_parquet(part, index=False)
    print(lang, len(rows), "rows ->", part)

In [ ]:
# Combine -> silver_pairs.parquet + results zip
import results_io
silver = pd.concat([pd.read_parquet(f"{PARTS}/{l}.parquet") for l in LANGS], ignore_index=True)
silver.to_parquet(f"{OUT}/silver_pairs.parquet", index=False)
report = {
    "rows": len(silver),
    "by_language_task": {f"{l}|{t}": int(n) for (l, t), n in silver.groupby(["language", "task"]).size().items()},
    "by_split": silver["split"].value_counts().to_dict(),
    "per_task": PER_TASK,
    "examples": silver.groupby("language").head(3)[["language", "task", "en_source_text", "source_text",
                                                    "en_target_text", "target_text"]].to_dict("records"),
}
results_io.write_json(OUT, "metrics.json", report)
results_io.write_config(OUT, {"per_task": PER_TASK, "langs": LANGS, "model": "ai4bharat/indictrans2-en-indic-dist-200M"},
                        f"{DATA}/manifest.json", CODE)
results_io.write_env(OUT, STARTED)
results_io.zip_results(OUT, f"{OUT}/civitas_results_silver.zip")
print({k: report[k] for k in ("rows", "by_language_task")})
print("download:", f"{OUT}/silver_pairs.parquet", "and", f"{OUT}/civitas_results_silver.zip")